# 📊 Exploration du Dataset Fashionpedia

Ce notebook vous permet d'explorer le dataset avant l'entraînement.

## 1. Installation et imports

In [ ]:
import sys
from pathlib import Path

# Ajouter le répertoire parent au path
BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, load_from_disk
import json
from PIL import Image, ImageDraw
from collections import Counter

# Configuration matplotlib
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style('whitegrid')

print("✅ Imports réussis")

## 2. Chargement du dataset

In [ ]:
# Vérifier si le dataset est déjà téléchargé
DATA_RAW = BASE_DIR / "data" / "raw"

if DATA_RAW.exists() and (DATA_RAW / "train").exists():
    print("📂 Chargement du dataset local...")
    train_dataset = load_from_disk(str(DATA_RAW / "train"))
    val_dataset = load_from_disk(str(DATA_RAW / "validation"))
    test_dataset = load_from_disk(str(DATA_RAW / "test"))
else:
    print("📥 Téléchargement du dataset depuis HuggingFace...")
    dataset = load_dataset("detection-datasets/fashionpedia_4_categories")
    train_dataset = dataset['train']
    val_dataset = dataset['validation']
    test_dataset = dataset['test']

print(f"\n✅ Dataset chargé")
print(f"   Train: {len(train_dataset)} images")
print(f"   Validation: {len(val_dataset)} images")
print(f"   Test: {len(test_dataset)} images")

## 3. Inspection de la structure

In [ ]:
# Examiner un exemple
sample = train_dataset[0]

print("📝 Structure d'un exemple :")
for key, value in sample.items():
    if key == 'image':
        print(f"  - {key}: PIL Image ({value.size})")
    elif key == 'objects':
        print(f"  - {key}: {len(value['bbox_id'])} objets")
    else:
        print(f"  - {key}: {value}")

## 4. Statistiques des catégories

In [ ]:
# Mapper les catégories
CATEGORIES = {
    0: "Accessories (Accessoires)",
    1: "Bags (Sacs)",
    2: "Clothing (Vêtements)",
    3: "Shoes (Chaussures)"
}

# Compter les objets par catégorie
def count_categories(dataset):
    category_counts = Counter()
    
    for sample in dataset:
        categories = sample['objects']['category']
        category_counts.update(categories)
    
    return category_counts

train_counts = count_categories(train_dataset)

print("📊 Répartition des catégories dans le train set :")
for cat_id, count in sorted(train_counts.items()):
    cat_name = CATEGORIES.get(cat_id, f"Unknown {cat_id}")
    percentage = count / sum(train_counts.values()) * 100
    print(f"  {cat_name:35s}: {count:6d} ({percentage:5.2f}%)")

## 5. Visualisation des catégories

In [ ]:
# Graphique en barres
fig, ax = plt.subplots(figsize=(10, 6))

categories_names = [CATEGORIES[i] for i in sorted(train_counts.keys())]
counts = [train_counts[i] for i in sorted(train_counts.keys())]

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars = ax.bar(categories_names, counts, color=colors)

ax.set_xlabel('Catégories', fontsize=12, fontweight='bold')
ax.set_ylabel('Nombre d\'objets', fontsize=12, fontweight='bold')
ax.set_title('Répartition des objets par catégorie (Train Set)', 
             fontsize=14, fontweight='bold')

# Ajouter les valeurs sur les barres
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Visualisation d'exemples avec bounding boxes

In [ ]:
def visualize_sample(sample, categories_dict):
    """
    Visualise une image avec ses bounding boxes
    """
    image = sample['image']
    objects = sample['objects']
    
    # Créer une copie pour dessiner
    draw_image = image.copy()
    draw = ImageDraw.Draw(draw_image)
    
    # Couleurs pour chaque catégorie
    colors = {
        0: '#FF6B6B',  # Rouge
        1: '#4ECDC4',  # Turquoise
        2: '#45B7D1',  # Bleu
        3: '#FFA07A'   # Orange
    }
    
    # Dessiner les bounding boxes
    for bbox, category in zip(objects['bbox'], objects['category']):
        x_min, y_min, x_max, y_max = bbox
        color = colors.get(category, '#FFFFFF')
        
        # Dessiner le rectangle
        draw.rectangle([x_min, y_min, x_max, y_max], 
                      outline=color, width=3)
        
        # Ajouter le label
        label = categories_dict.get(category, f"Cat {category}")
        draw.text((x_min, y_min - 20), label, fill=color)
    
    return draw_image

# Visualiser plusieurs exemples
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, ax in enumerate(axes):
    # Prendre un échantillon aléatoire
    idx = np.random.randint(0, len(train_dataset))
    sample = train_dataset[idx]
    
    # Visualiser
    annotated_image = visualize_sample(sample, CATEGORIES)
    
    ax.imshow(annotated_image)
    ax.set_title(f"Image {sample['image_id']} - {len(sample['objects']['bbox_id'])} objets",
                fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.suptitle('Exemples d\'images avec annotations', 
             fontsize=14, fontweight='bold', y=1.02)
plt.show()

## 7. Statistiques des tailles d'images

In [ ]:
# Collecter les dimensions
widths = [sample['width'] for sample in train_dataset]
heights = [sample['height'] for sample in train_dataset]
aspects = [w/h for w, h in zip(widths, heights)]

print("📐 Statistiques des dimensions :")
print(f"   Largeur  - Min: {min(widths)}, Max: {max(widths)}, Moyenne: {np.mean(widths):.0f}")
print(f"   Hauteur  - Min: {min(heights)}, Max: {max(heights)}, Moyenne: {np.mean(heights):.0f}")
print(f"   Ratio W/H - Min: {min(aspects):.2f}, Max: {max(aspects):.2f}, Moyenne: {np.mean(aspects):.2f}")

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(widths, bins=50, color='#45B7D1', edgecolor='black')
axes[0].set_title('Distribution des largeurs', fontweight='bold')
axes[0].set_xlabel('Largeur (pixels)')
axes[0].set_ylabel('Fréquence')

axes[1].hist(heights, bins=50, color='#FF6B6B', edgecolor='black')
axes[1].set_title('Distribution des hauteurs', fontweight='bold')
axes[1].set_xlabel('Hauteur (pixels)')
axes[1].set_ylabel('Fréquence')

axes[2].hist(aspects, bins=50, color='#4ECDC4', edgecolor='black')
axes[2].set_title('Distribution des ratios W/H', fontweight='bold')
axes[2].set_xlabel('Ratio largeur/hauteur')
axes[2].set_ylabel('Fréquence')

plt.tight_layout()
plt.show()

## 8. Nombre d'objets par image

In [ ]:
# Compter les objets par image
objects_per_image = [len(sample['objects']['bbox_id']) for sample in train_dataset]

print("🔢 Statistiques du nombre d'objets par image :")
print(f"   Minimum : {min(objects_per_image)}")
print(f"   Maximum : {max(objects_per_image)}")
print(f"   Moyenne : {np.mean(objects_per_image):.2f}")
print(f"   Médiane : {np.median(objects_per_image):.0f}")

# Visualisation
plt.figure(figsize=(10, 6))
plt.hist(objects_per_image, bins=range(0, max(objects_per_image)+2), 
         color='#FFA07A', edgecolor='black', alpha=0.7)
plt.axvline(np.mean(objects_per_image), color='red', 
            linestyle='--', linewidth=2, label=f'Moyenne: {np.mean(objects_per_image):.2f}')
plt.xlabel('Nombre d\'objets par image', fontsize=12, fontweight='bold')
plt.ylabel('Fréquence', fontsize=12, fontweight='bold')
plt.title('Distribution du nombre d\'objets par image', 
          fontsize=14, fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Mapping vers le code vestimentaire ENSITECH

In [ ]:
# Mapping des catégories
CATEGORY_MAPPING = {
    0: {"id": 0, "name": "accessoire_interdit", 
        "description": "Casquette, chapeau, bonnet, bandana"},
    1: {"id": None, "name": "sac", 
        "description": "Sac (non concerné)"},
    2: {"id": 1, "name": "vetement_interdit", 
        "description": "Crop top, dos ouvert, tenue sport, short, etc."},
    3: {"id": 2, "name": "chaussure_interdite", 
        "description": "Tongs"}
}

# Compter les objets pertinents pour le code vestimentaire
relevant_counts = Counter()
ignored_count = 0

for sample in train_dataset:
    for cat in sample['objects']['category']:
        mapped = CATEGORY_MAPPING[cat]
        if mapped['id'] is not None:
            relevant_counts[mapped['name']] += 1
        else:
            ignored_count += 1

print("🏷️  Objets pertinents pour le code vestimentaire :")
total_relevant = sum(relevant_counts.values())
for name, count in relevant_counts.items():
    percentage = count / total_relevant * 100
    print(f"   {name:25s}: {count:6d} ({percentage:5.2f}%)")

print(f"\n   Objets ignorés (sacs): {ignored_count:6d}")
print(f"   Total pertinent      : {total_relevant:6d}")

## 10. Résumé et conclusions

In [ ]:
print("=" * 60)
print("📊 RÉSUMÉ DE L'EXPLORATION")
print("=" * 60)

print(f"\n1. Dataset :")
print(f"   - Total d'images : {len(train_dataset) + len(val_dataset) + len(test_dataset):,}")
print(f"   - Train : {len(train_dataset):,} (90%)")
print(f"   - Validation : {len(val_dataset):,} (5%)")
print(f"   - Test : {len(test_dataset):,} (5%)")

print(f"\n2. Catégories pour le code vestimentaire :")
print(f"   - 3 catégories pertinentes")
print(f"   - {total_relevant:,} objets pertinents")

print(f"\n3. Dimensions des images :")
print(f"   - Taille moyenne : {np.mean(widths):.0f} x {np.mean(heights):.0f} pixels")
print(f"   - YOLO utilisera : 640 x 640 pixels")

print(f"\n4. Objets par image :")
print(f"   - Moyenne : {np.mean(objects_per_image):.1f} objets/image")

print(f"\n5. Prochaines étapes :")
print(f"   ✅ Dataset exploré et compris")
print(f"   ⏭️  Préparer les données pour YOLO")
print(f"   ⏭️  Entraîner le modèle")

print("\n" + "=" * 60)